In [1]:
from dotenv import load_dotenv
import os
import json
import pandas as pd
from datetime import datetime
from google.oauth2 import service_account
from googleapiclient.discovery import build
from google import genai
import requests

load_dotenv()

TG_BOT_TOKEN = os.environ["TG_BOT_TOKEN"]
TG_GROUP_ID = os.environ["TG_GROUP_ID"]
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]
GOOGLE_SA_JSON = json.loads(os.environ["GOOGLE_SA_JSON"])
GOOGLE_ID = os.environ["GOOGLE_ID"]

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

In [3]:
credentials = service_account.Credentials.from_service_account_info(
    GOOGLE_SA_JSON,
    scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"],
)

In [4]:
service = build("sheets", "v4", credentials=credentials)
result = service.spreadsheets().values().get(
    spreadsheetId=GOOGLE_ID,
    range="bd!A1:E",  # или "Sheet1!A1:Z" / имя своей вкладки
).execute()

values = result.get("values", [])
df = pd.DataFrame(values[1:], columns=values[0])

In [5]:
df['day'] = df.birth_date.str[0:2].astype(int) 
df['month'] = df.birth_date.str[3:5].astype(int)

day = datetime.today().day
month = datetime.today().month

In [6]:
df_bd = df.query('day == @day and month == @month and active == "TRUE"')

In [ ]:
for row in df_bd.itertuples():
    try:
        name = row.name
        notes = row.notes
        card = row.card

        response = gemini_client.models.generate_content(
            model="gemini-3-flash-preview",
            contents=f"""Сгенерируй короткое поздравление с днём рождения.

Имя: {name}
Заметки: {notes}

Требования:
- 2–3 предложения
- обращение по имени, на «ты»
- учти заметки (общие интересы, шутки, контекст отношений)
- без клише ("счастья, здоровья, успехов")
- если в заметках есть негативные характеристики человека — обыграй их по-доброму, без сарказма
- максимум 3 эмодзи на всё поздравление (или ни одного)
- можно выделить *курсивом* или жирным **шрифтом** фразу или одно слово-акцент, если уместно
- эмодзи только если он реально к месту (по интересам из заметок), не для "украшения"

Верни только текст поздравления.""",
        )

        greeting = response.text.strip()
        card_line = f"\n\nДля тех кто хочет поздравить: {card}" if pd.notna(card) else ""
        full_message = f"#birthday\n{greeting}{card_line}"

        # отправка в Telegram
        r = requests.post(
            f"https://api.telegram.org/bot{TG_BOT_TOKEN}/sendMessage",
            json={
                "chat_id": TG_GROUP_ID,
                "text": full_message,
                "parse_mode": "Markdown",
            },
        )
        print(f"{'✅' if r.ok else '❌'} {name}: {r.status_code}")
        if not r.ok:
            print(r.json())

    except Exception as e:
        print(f"❌ {name}: {e}")


✅ Ким Денис: 200
✅ Адреан Бета: 200
